# # ML - 지도학습 - Timeseries Forecasting 실습

[사용 제한 안내]
> 본 실습 코드는 교육 및 학습 목적으로만 제공됩니다.

[사전 준비 사항]
- 없음

[분석 요건]
- 분석 주제 : 머신러닝 기법을 활용한 시계열 예측 모델 개발
- 분석 단위 : month
- 분석 변수 : 월별 승객수

[주요 실습 내용]
- Timeseries 데이터 특성 확인
- 시계열 예측에서 Target / Feature 데이타 구조
- Timeseries 데이타 파생변수 생성 : Lag 변수 등
- ML 이용한 Timeseries 예측 모델 구조

# 데이타 준비
- 분석 데이타 : Airline Passengers Dataset
- 데이타 설명 : 월별 항공 승객 수 데이타를 활용하여 미래 승객 수를 예측하기 위함
- 기간 : 1949년 1월 ~ 1960년 12월, 144개월(12년)
- 변수 설명
| 컬럼명        | 한글명     | 의미                         | 비고              |
| ---------- | ------- | -------------------------- | --------------- |
| Month      | 기준 월    | 연도-월(YYYY-MM)              | 문자형 |
| Passengers | 항공 승객 수 | 해당 월 국제선 항공 승객 수 (단위: 천 명) | Target 변수       |







데이타 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url)
df

컬럼정보 확인

In [ ]:
df.info()

[참고] info()에서 확인할 것:
  - 총 행/열 수
  - 각 컬럼의 데이터 타입
    - int : 연속형 (정수형)
    - float : 연속형 (실수형)
    - object : 보통 문자열 (범주형)
    - category : 범주형,  *범주 갯수가 적을 경우 메모리 효율화를 위해 object 타입 대신 사용, 그러나 object 을 사용해도 무방
    - datetime : 날짜형fo()

데이타 전처리

In [ ]:
# 컬럼명 변경
df.columns = ["date", "passengers"]

# 날짜형 변환
df["date"] = pd.to_datetime(df["date"])

# 날짜 기준 정렬
df = df.sort_values("date").reset_index(drop=True)
df

# EDA

연속형 변수 기본 분포값 확인
 - 비즈니스관점에서 데이터 이상 여부 확인, 특히 음수(-), '0' 데이터 주의
 - Missing 건수 확인
 - 데이터 최소, 최대, 중심값 확인
 - 데이터 밀집도 확인

In [ ]:
import numpy as np

# 연속형 변수 기본 분포 확인
df.select_dtypes(include=[np.number]).describe()

# Feature Creation

- Timeseries 파생변수 생성

In [ ]:
# 날짜 변수
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

# 과거 Lag 변수
df["lag_1"] = df["passengers"].shift(1)     # 1개월 전
df["lag_2"] = df["passengers"].shift(2)     # 2개월 전
df["lag_3"] = df["passengers"].shift(3)     # 3개월 전
df["lag_12"] = df["passengers"].shift(12)   # 1년전

# 이동평균 변수
df["rolling_mean_3"] = df["passengers"].rolling(window=3).mean()        # 3개월 평균
df["rolling_mean_6"] = df["passengers"].rolling(window=6).mean()        # 6개월 평균
df["rolling_mean_12"] = df["passengers"].rolling(window=12).mean()      # 12개월 평균

# 파생변수 미생성건 제거
df_model = df.dropna().reset_index(drop=True)
df_model

# Target 설정

In [ ]:
# 예측 기간
look_after = 1

# Target 설정: 다음 월(t+1) 승객 수
df_model["target"] = df_model["passengers"].shift(look_after*(-1))

# Target 미생성건 제거
df_model = df_model.dropna().reset_index(drop=True)
df_model

# 데이타 분할

Data Partition
- 목적 : 모델이 학습에 사용하지 않은 새로운 데이터에서도 잘 동작하는지 검증하기 위함
 - 시간 순서를 유지하여 과거 데이터로 학습 하고 (Train), 미래 데이터로 평가 (Test)
- Train Set : 모델 학습용, Test Set : 모델 평가용
- 데이타 변환시 변환 통계량은 Train 데이터에서 계산하고, Test 데이터에는 그 기준을 적용해야 함 (Data Leakage 문제)

In [ ]:
# 마지막 2년을 테스트셋으로 사용
train = df_model.iloc[:-24]
test = df_model.iloc[-24:]

X, y 분리

In [ ]:
# train
y_train  = train['target']
X_train  = train.drop(columns=['target', 'date'])

# test
y_test   = test['target']
X_test   = test.drop(columns=['target', 'date'])

# Feature Transformation

Scaling
   - Min-Max Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 연속형 변수 추출
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Min-Max Scaling
scaler = MinMaxScaler()

# Train
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

 # Test
X_test[num_cols] = scaler.transform(X_test[num_cols])

# Random Forest
  - API 문서 : https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html

학습 및 예측

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# 모델 정의
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

# 모델 학습
model.fit(X_train, y_train)

# 예측
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# 평가
print("\n=== RandomForest === ")
print(f"Train MAPE : {mean_absolute_percentage_error(y_train, train_pred) * 100:.2f}%")
print(f"Test  MAPE : {mean_absolute_percentage_error(y_test, test_pred) * 100:.2f}%")

- Feature Importance

In [ ]:
# Feature Importance
importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print(importance)

# 시각화
plt.figure(figsize=(8, 4))
plt.barh(importance["Feature"], importance["Importance"], color="teal")
plt.gca().invert_yaxis()
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

- 원래 데이터에 예측값 병합하기

In [ ]:
# Test Set 에 예측값 병합하기
result = test[["date", "passengers", "target"]].copy()
result["prediction_t_plus_1"] = test_pred

result = result.rename(columns={
    "date": "current_month",
    "passengers": "current_passengers",
    "target": "actual_next_month"
})

result

- 실측값 vs 예측값 시각화

In [ ]:
# Test Set 기준 실측값 vs 예측값 시각화
plt.figure(figsize=(10, 4))

plt.plot(
    result["current_month"],
    result["actual_next_month"],
    marker="o",
    label="Actual Next Month"
)

plt.plot(
    result["current_month"],
    result["prediction_t_plus_1"],
    marker="o",
    label="Predicted Next Month"
)

plt.title("Next Month Air Passengers Forecast")
plt.xlabel("Current Month")
plt.ylabel("Passengers")
plt.legend()
plt.grid(True)
plt.show()

### [Self-Practice]
Q1) 다른 ML 모델들을 적용해서 모델 성능을 비교해 보세요. 예시) LinearRegression, XGBRegressor, LGBMRegressor 등  
Q2) 성능이 가장 좋은 모델로 Test 예측값을 생성하고, 시각화를 통해 예측값과 실측값을 비교해 보세요.  
Q3) 성능이 가장 좋은 모델기준으로 시계열 파생변수를 추가하거나 제거하면서 모델 성능을 비교하세요.   
Q4) 성능이 가장 좋은 모델기준으로 다음 달 (t+1) 예측이 아니라 더 먼 미래 (t+3), (t+5)를 예측하는 모델을 생성하고, 각 예측기간별 모델 성능을 비교해 보세요.   